In [ ]:
pip install pandas torch torchvision torchaudio transformers tokenizers

In [ ]:
pip install sentencepiece

In [ ]:
!pip install transformers huggingface_hub

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

text = [
    "Prompt : Hi | Response A : Hello | Response B : Hey there",
    "Prompt : Bye | Response A : See ya | Response B : Goodbye"
]

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large", use_fast=True)

# Note: Removed return_overflowing_tokens=True to prevent batch size mismatches with labels
result = tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt")

input_ids = result['input_ids'].to(device)
attention_mask = result['attention_mask'].to(device)

# 3 target classes (e.g., Class 0, Class 1, Class 2)
labels = torch.tensor([0, 2], dtype=torch.long).to(device)

# Initialize model
model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-large", num_labels=3).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-6) # Lower LR for stability

output = model(input_ids, attention_mask)
print(f"the output of the function is : {output}")

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifie

the output of the function is : SequenceClassifierOutput(loss=None, logits=tensor([[-0.1076, -0.2844,  0.2096],
        [-0.1063, -0.2849,  0.2164]], device='cuda:0', dtype=torch.float16,
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

class CustomDebertaClassifier(nn.Module):
  def __init__ (self,model_name = "microsoft/deberta-v3-large", num_labels = 3) :
    super().__init__()
    self.deberta = AutoModel.from_pretrained(model_name)
    hidden_size = self.deberta.config.hidden_size
    intermediate_size = 256
    self.dense = nn.Linear(hidden_size, intermediate_size)
    self.activation = nn.GELU()

    self.dropout= nn.Dropout(p=0.1)
    self.classifier = nn.Linear(intermediate_size,num_labels)
  def forward (self,input_ids, attention_mask):
    outputs = self.deberta(input_ids = input_ids, attention_mask = attention_mask)
    hidden_states = outputs.last_hidden_state

    input_mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
    sum_embeddings = torch.sum(hidden_states * input_mask_expanded, 1)
    sum_mask = input_mask_expanded.sum(1)
    sum_mask = torch.clamp(sum_mask, min=1e-9)
    pooled_output = sum_embeddings / sum_mask

    x= self.dense (pooled_output)
    x = self.activation(x)
    x= self.dropout(x)
    logits = self.classifier(x)
    return logits

model = CustomDebertaClassifier(num_labels = 3).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-6, eps=1e-6)
output = model(input_ids, attention_mask)

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
labels = torch.tensor([0,2], dtype = torch.long).to(device)
epochs = 1000
for epoch in range(epochs):
  model.train()
  logits = model(input_ids, attention_mask)
  loss = loss_fn(logits, labels)
  optimizer.zero_grad()
  loss.backward()
  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
  optimizer.step()

  if epoch%100 ==0 :
    print(f"the loss is : {loss}")


the loss is : 0.7605462670326233
the loss is : 0.7309025526046753
the loss is : 0.8758043646812439
the loss is : 0.6224453449249268
the loss is : 0.6874180436134338
the loss is : 0.6816138029098511
the loss is : 0.7345811128616333
the loss is : 0.6012258529663086
the loss is : 0.8056199550628662
the loss is : 0.786971926689148
